In [1]:
import numpy as np
import config
from src.data.sudoku_gen import count_solutions

for g in config.GIVENS_LEVELS:
    d = np.load(config.SUDOKU_DIR / f"puzzles_g{g}.npz")
    P, S = d["puzzles"], d["solutions"]
    assert P.shape == (100, 81)
    assert ((P != 0).sum(1) == g).all()                  # exact givens
    assert ((P == 0) | (P == S)).all()                   # givens agree with solution
    assert all(count_solutions(p, 2) == 1 for p in P)    # unique
    assert len(set(d["puzzle_seeds"])) == 100            # distinct seeds
    print(g, "OK", np.bincount(d["seed_group"]))         # expect [20 20 20 20 20]


25 OK [20 20 20 20 20]
30 OK [20 20 20 20 20]
35 OK [20 20 20 20 20]


In [2]:
import numpy as np, config
from make_data import puzzle_seed
from src.data.sudoku_gen import make_puzzle

d = np.load(config.SUDOKU_DIR / "puzzles_g25.npz")
p, s = make_puzzle(25, puzzle_seed(25, 7))
print((p == d["puzzles"][7]).all(), (s == d["solutions"][7]).all())   # True True

True True


In [3]:
from src.data.mosaic import load_raw_test, render_mosaic, split_tiles, blank_mask

test_images, test_labels = load_raw_test()
for g in config.GIVENS_LEVELS:
    d = np.load(config.SUDOKU_DIR / f"puzzles_g{g}.npz")
    P, I = d["puzzles"], d["cell_img_idx"]
    assert I.shape == (100, 81) and I.dtype == np.int32
    assert ((I == -1) == (P == 0)).all()                       # -1 exactly on blanks
    given = P != 0
    assert (test_labels[I[given]] == P[given]).all()           # image label == cell digit
    assert all(len(set(r[r >= 0])) == (r >= 0).sum() for r in I)   # no repeats in a puzzle
    for k in (0, 50, 99):                                      # mosaic round-trip + blank rule
        tiles = split_tiles(render_mosaic(I[k], test_images))
        assert (blank_mask(tiles) == (P[k] == 0)).all()
    print(g, "mosaic OK")

25 mosaic OK
30 mosaic OK
35 mosaic OK
